# Guía 3 V3 - Aprendizaje supervisado

Este notebook está diseñado para comprender, no solo ejecutar. Cada bloque incluye propósito, código, análisis y conclusión.

## 1. Contexto
El objetivo es predecir `Abandono` usando variables predictoras. Antes de entrenar se debe separar `X` e `y`, excluir identificadores y evitar fuga de información.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

df = pd.read_csv('base.csv')
df.head()

,ID_Cliente,Edad,IngresoMensual,CantidadCompras,ComprasUltimos12M,AntiguedadMeses,QuejasUltimos6M,DiasDesdeUltimaCompra,VisitasWebUltimoMes,TiempoPromedioSesionMin,CuponesUsados,Ciudad,CanalPreferido,ZonaResidencia,Segmento,Satisfaccion,CodigoCampania,Abandono
0,CLI_0001,18,3968,17,5,4,0,39,11,7.72,1,Cali,Tienda,Urbana,Básico,Media,CAMP_10,0
1,CLI_0002,50,3528,18,8,53,1,12,1,5.50,1,Medellin,Telefono,Urbana,Básico,Alta,CAMP_04,0
2,CLI_0003,46,750,16,8,90,1,83,10,6.15,3,Cartagena,Web,Urbana,Básico,Alta,CAMP_09,0
3,CLI_0004,20,4356,21,4,25,3,85,4,10.88,2,Cartagena,Tienda,Rural,Medio,Baja,CAMP_06,1
4,CLI_0005,61,6155,12,4,10,0,21,6,0.51,2,Bogota,Tienda,Urbana,Medio,Alta,CAMP_01,0


### Análisis esperado
Revise las primeras filas para reconocer variables numéricas, categóricas, ordinales y la variable objetivo. No se toman decisiones todavía; primero se comprende la estructura.

In [3]:
df.info()
print('Filas y columnas:', df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID_Cliente               420 non-null    object 
 1   Edad                     420 non-null    int64  
 2   IngresoMensual           420 non-null    int64  
 3   CantidadCompras          420 non-null    int64  
 4   ComprasUltimos12M        420 non-null    int64  
 5   AntiguedadMeses          420 non-null    int64  
 6   QuejasUltimos6M          420 non-null    int64  
 7   DiasDesdeUltimaCompra    420 non-null    int64  
 8   VisitasWebUltimoMes      420 non-null    int64  
 9   TiempoPromedioSesionMin  420 non-null    float64
 10  CuponesUsados            420 non-null    int64  
 11  Ciudad                   420 non-null    object 
 12  CanalPreferido           420 non-null    object 
 13  ZonaResidencia           420 non-null    object 
 14  Segmento                 4

## 2. Variable objetivo
La variable objetivo es `Abandono`. El modelo intentará aprender patrones que permitan clasificar clientes en 0=no abandono y 1=abandono.

In [4]:
objetivo = 'Abandono'
print(df[objetivo].value_counts())
print(df[objetivo].value_counts(normalize=True).round(4))

Abandono
0    240
1    180
Name: count, dtype: int64
Abandono
0    0.5714
1    0.4286
Name: proportion, dtype: float64


### Interpretación
Este conteo no soluciona desbalance: solo lo diagnostica. Si una clase domina demasiado, accuracy puede ser engañoso. En abandono interesa especialmente detectar la clase 1, por eso recall de la clase positiva será una métrica importante.

## 3. Separar X e y
`X` contiene variables predictoras. `y` contiene la respuesta. `ID_Cliente` se excluye porque identifica el registro, pero no describe comportamiento.

In [5]:
X = df.drop(columns=['Abandono','ID_Cliente'])
y = df['Abandono']
print(X.columns.tolist())
print(y.head())

['Edad', 'IngresoMensual', 'CantidadCompras', 'ComprasUltimos12M', 'AntiguedadMeses', 'QuejasUltimos6M', 'DiasDesdeUltimaCompra', 'VisitasWebUltimoMes', 'TiempoPromedioSesionMin', 'CuponesUsados', 'Ciudad', 'CanalPreferido', 'ZonaResidencia', 'Segmento', 'Satisfaccion', 'CodigoCampania']
0    0
1    0
2    0
3    1
4    0
Name: Abandono, dtype: int64


## 4. División entrenamiento/prueba
Se usa `stratify=y` para conservar proporciones similares de abandono/no abandono en entrenamiento y prueba.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Distribución train')
print(y_train.value_counts(normalize=True).round(4))
print('Distribución test')
print(y_test.value_counts(normalize=True).round(4))

Train: (336, 16) Test: (84, 16)
Distribución train
Abandono
0    0.5714
1    0.4286
Name: proportion, dtype: float64
Distribución test
Abandono
0    0.5714
1    0.4286
Name: proportion, dtype: float64


### Conclusión parcial
El conjunto de prueba queda reservado para evaluación. No se debe usar para ajustar escaladores, codificadores ni modelos.

## 5. Preprocesamiento correcto
Segmento y Satisfaccion son ordinales. Ciudad, CanalPreferido, ZonaResidencia y CodigoCampania son nominales. Las variables numéricas se escalan para modelos sensibles a escala.

In [7]:
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
ord_cols = ['Segmento','Satisfaccion']
nom_cols = ['Ciudad','CanalPreferido','ZonaResidencia','CodigoCampania']

segmento_orden = ['Básico','Medio','Premium']
satisfaccion_orden = ['Baja','Media','Alta']

preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('ord', OrdinalEncoder(categories=[segmento_orden, satisfaccion_orden]), ord_cols),
        ('nom', OneHotEncoder(handle_unknown='ignore'), nom_cols)
    ]
)
print('Numéricas:', num_cols)
print('Ordinales:', ord_cols)
print('Nominales:', nom_cols)

Numéricas: ['Edad', 'IngresoMensual', 'CantidadCompras', 'ComprasUltimos12M', 'AntiguedadMeses', 'QuejasUltimos6M', 'DiasDesdeUltimaCompra', 'VisitasWebUltimoMes', 'TiempoPromedioSesionMin', 'CuponesUsados']
Ordinales: ['Segmento', 'Satisfaccion']
Nominales: ['Ciudad', 'CanalPreferido', 'ZonaResidencia', 'CodigoCampania']


## 6. Entrenar modelos básicos
Se comparan tres modelos. Todos usan el mismo preprocesamiento para que la comparación sea justa.

In [8]:
modelos = {
    'Regresión logística': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Árbol de decisión': DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'),
    'Bosque aleatorio': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight='balanced')
}

resultados = []
for nombre, modelo in modelos.items():
    pipe = Pipeline(steps=[('preprocess', preprocess), ('model', modelo)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    resultados.append({
        'modelo': nombre,
        'accuracy': accuracy_score(y_test, pred),
        'precision_clase_1': precision_score(y_test, pred, zero_division=0),
        'recall_clase_1': recall_score(y_test, pred, zero_division=0),
        'f1_clase_1': f1_score(y_test, pred, zero_division=0)
    })

res = pd.DataFrame(resultados).sort_values('f1_clase_1', ascending=False)
res

,modelo,accuracy,precision_clase_1,recall_clase_1,f1_clase_1
0,Regresión logística,0.619048,0.547619,0.638889,0.589744
1,Árbol de decisión,0.559524,0.489796,0.666667,0.564706
2,Bosque aleatorio,0.583333,0.513514,0.527778,0.520548


### Análisis de resultados
No se elige automáticamente el modelo con mayor accuracy. En abandono, es importante observar recall de la clase 1, porque representa la capacidad de detectar clientes que sí abandonan. F1 ayuda a equilibrar precision y recall.

In [9]:
mejor_nombre = res.iloc[0]['modelo']
print('Modelo con mejor F1 para clase 1:', mejor_nombre)

Modelo con mejor F1 para clase 1: Regresión logística


## 7. Matriz de confusión del modelo seleccionado
La matriz permite entender qué tipos de errores comete el modelo.

In [10]:
modelo_final = modelos[mejor_nombre]
pipe_final = Pipeline(steps=[('preprocess', preprocess), ('model', modelo_final)])
pipe_final.fit(X_train, y_train)
y_pred = pipe_final.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

[[29 19]
 [13 23]]
              precision    recall  f1-score   support

           0       0.69      0.60      0.64        48
           1       0.55      0.64      0.59        36

    accuracy                           0.62        84
   macro avg       0.62      0.62      0.62        84
weighted avg       0.63      0.62      0.62        84



### Interpretación final
Revise falsos negativos y falsos positivos. Un falso negativo significa que un cliente que sí abandonó no fue detectado. Esa clase de error puede ser crítica si el objetivo del negocio es retener clientes.

## 8. Conclusión técnica para el informe
El modelo debe presentarse como apoyo a la toma de decisiones. Se deben declarar métricas, limitaciones, datos usados, posibles sesgos y necesidad de revisión humana.

## Actividad del aprendiz
Complete el análisis escrito: ¿qué modelo selecciona y por qué? ¿Qué métrica prioriza? ¿Qué limitaciones observa?